# 16.1 — BCG mouse preprocessing (atlas HVG **intersection**, no zero-fill)

Same pipeline as **notebook 16** (raw counts → symbol → ortholog → per-flavor atlas gene list → `normalize_total` + `log1p` → CellOT `_v07` rewrite), with one important change:

**Notebook 16** forced **exactly 1,000 columns** matching the atlas `var_names` order and **filled missing genes with zeros** (~47–50% of columns for Seurat HVG were artificial zeros). That keeps matrix shape aligned with a 1k-gene atlas but misrepresents unmeasured genes.

**Notebook 16.1** instead keeps **only atlas HVG genes that are actually present in BCG** (after symbol→ENSMUSG→ENSG mapping). Columns are **in atlas HVG order**; `n_vars` is the intersection size (**~530–650** in typical runs), **no zero-padding**.

**Implication for models**: a CellOT/scGen checkpoint trained on **(n_cells, 1000)** atlas data **will not** accept these BCG matrices without **also subsetting the atlas (or the model)** to the **same gene set**. Prefer storing a shared gene list CSV per flavor next to the outputs.

**Outputs** (do not overwrite notebook 16 files):
- `bcg_mouse_atlashvg_intersect_{flavor}.h5ad`
- `bcg_mouse_atlashvg_intersect_{flavor}_v07.h5ad`
- `bcg_atlas_hvg_coverage_intersection.csv`

**Dependency**: notebook **15** for `hvg_{flavor}_atlas_full_v07.h5ad` and ortholog cache.


In [1]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp_sparse

sys.path.insert(0, "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT")

BASE_DIR = "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT"
BCG_PATH = "/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/tb/data/bcg_mouse_aligned_050626.h5ad"
DATASET_DIR = os.path.join(BASE_DIR, "cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg")
ORTHO_CACHE = os.path.join(BASE_DIR, "scripts/.biomart_ortholog_cache.csv")
SYMBOL_CACHE = os.path.join(BASE_DIR, "scripts/.bcg_symbol_to_ensmusg.csv")
OUT_DIR = os.path.join(BASE_DIR, "speciesOT/baseline/analysis/bcg_mouse_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

FLAVORS = ["seurat_v3", "pearson_residuals"]
CELLOT_PY = "/n/home01/jzhou1125/.conda/envs/CellOT/bin/python"

print("BCG source:", BCG_PATH)
print("ortholog cache:", ORTHO_CACHE, "(exists?", os.path.exists(ORTHO_CACHE), ")")
print("symbol cache:", SYMBOL_CACHE, "(exists?", os.path.exists(SYMBOL_CACHE), ")")
print("dataset dir:", DATASET_DIR)
print("flavors:", FLAVORS)

BCG source: /n/holylabs/mooney_lab/Lab/joshprice/speciesOT/tb/data/bcg_mouse_aligned_050626.h5ad
ortholog cache: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/scripts/.biomart_ortholog_cache.csv (exists? True )
symbol cache: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/scripts/.bcg_symbol_to_ensmusg.csv (exists? False )
dataset dir: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg
flavors: ['seurat_v3', 'pearson_residuals']


## 1. Load BCG mouse, swap raw counts into `.X`

In [2]:
bcg = sc.read_h5ad(BCG_PATH)
print("BCG raw read:", bcg.shape, "vars sample:", list(bcg.var_names[:5]))
print("obs cols:", list(bcg.obs.columns))
print("layers:", list(bcg.layers.keys()))

assert "counts" in bcg.layers, "expected raw counts in .layers['counts']"

# Replace .X with raw counts and ensure float32 for downstream consistency
bcg.X = bcg.layers["counts"].astype("float32")
del bcg.layers  # avoid carrying scanvi-corrupted layers around

# Make obs unique
bcg.obs_names_make_unique()

# Tag as 'mouse' for downstream condition handling
bcg.obs["condition"] = "mouse"
bcg.obs["species"] = "mouse"

print("\nAfter swap:")
print("  shape:", bcg.shape)
x_sample = bcg.X[:100].toarray().ravel() if sp_sparse.issparse(bcg.X) else bcg.X[:100].ravel()
print(f"  .X dtype={bcg.X.dtype}, sample min={x_sample.min():.0f}, max={x_sample.max():.0f}, mean={x_sample.mean():.2f}")
print(f"  all integer-valued? {np.allclose(x_sample, np.round(x_sample))}")
print(f"  cell_type counts: {bcg.obs['cell_type'].value_counts().to_dict()}")

BCG raw read: (1406, 10866) vars sample: ['Mrpl15', 'Lypla1', 'Tcea1', 'Atp6v1h', 'Rb1cc1']
obs cols: ['n_genes', 'leiden', 'cell_type', 'study', 'cell_type_original', 'cell_type_scanvi', '_scvi_batch', '_scvi_labels']
layers: ['counts']

After swap:
  shape: (1406, 10866)
  .X dtype=float32, sample min=0, max=457, mean=0.58
  all integer-valued? True
  cell_type counts: {'LT-HSC treated': 994, 'LT-HSC': 412}


/n/home01/jzhou1125/miniforge3/envs/analysis/lib/python3.12/site-packages/anndata/_core/anndata.py:1823: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


## 2. Map mouse symbols → mouse Ensembl IDs (ENSMUSG)

Use BioMart to map gene symbols. Cache to disk so we don't re-query if BioMart is flaky later.

In [3]:
def fetch_mouse_symbol_to_ensmusg(symbols, host="www.ensembl.org", max_attempts=3, pause_sec=2.0):
    """BioMart query: mouse gene symbols -> ENSMUSG IDs. Returns dict symbol -> ensmusg (one-to-one).
    For multi-mapped symbols, picks the first."""
    import time
    from pybiomart import Dataset
    hosts = [host, "useast.ensembl.org", "uswest.ensembl.org", "asia.ensembl.org"]
    last_err = None
    for h in hosts:
        for attempt in range(max_attempts):
            try:
                ds = Dataset(name="mmusculus_gene_ensembl", host=h)
                df = ds.query(attributes=["ensembl_gene_id", "external_gene_name"]).copy()
                df.columns = ["ensmusg", "symbol"]
                # Drop rows with NaN
                df = df.dropna(subset=["ensmusg", "symbol"])
                # Filter to symbols we care about (case-insensitive could be useful but skip for now)
                want = set(symbols)
                df = df[df["symbol"].isin(want)]
                # Take first ENSMUSG per symbol
                df = df.drop_duplicates("symbol", keep="first")
                return dict(zip(df["symbol"], df["ensmusg"]))
            except Exception as e:
                last_err = e
                time.sleep(pause_sec)
    raise RuntimeError(f"BioMart symbol query failed on all hosts. Last error: {last_err!r}")


bcg_symbols = list(bcg.var_names.astype(str))
print(f"BCG has {len(bcg_symbols)} gene symbols")

if os.path.exists(SYMBOL_CACHE):
    cache_df = pd.read_csv(SYMBOL_CACHE)
    sym2ensmusg = dict(zip(cache_df["symbol"], cache_df["ensmusg"]))
    print(f"  loaded {len(sym2ensmusg)} symbol->ENSMUSG mappings from cache")
else:
    print(f"  cache miss; querying BioMart...")
    sym2ensmusg = fetch_mouse_symbol_to_ensmusg(bcg_symbols)
    pd.DataFrame({"symbol": list(sym2ensmusg.keys()),
                  "ensmusg": list(sym2ensmusg.values())}).to_csv(SYMBOL_CACHE, index=False)
    print(f"  cached {len(sym2ensmusg)} mappings -> {SYMBOL_CACHE}")

n_mapped = sum(1 for s in bcg_symbols if s in sym2ensmusg)
print(f"\n  symbols with ENSMUSG mapping: {n_mapped} / {len(bcg_symbols)} ({n_mapped/len(bcg_symbols):.1%})")

BCG has 10866 gene symbols
  cache miss; querying BioMart...


  cached 10451 mappings -> /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/scripts/.bcg_symbol_to_ensmusg.csv

  symbols with ENSMUSG mapping: 10451 / 10866 (96.2%)


## 3. Use cached ortholog table to map ENSMUSG → ENSG (human)

In [4]:
assert os.path.exists(ORTHO_CACHE), (
    f"BioMart ortholog cache not found at {ORTHO_CACHE}. "
    f"Please run 15_data_prep_full_atlas_no_holdout.ipynb first to generate it."
)
ortho_df = pd.read_csv(ORTHO_CACHE)
print(f"Loaded ortholog table: {len(ortho_df)} rows")
print("  columns:", list(ortho_df.columns))

# Build dict ENSMUSG -> ENSG
# Filter to one2one orthology if column exists
if "orthology_type" in ortho_df.columns:
    ortho_df = ortho_df[ortho_df["orthology_type"] == "ortholog_one2one"]
    print(f"  one2one only: {len(ortho_df)} rows")

ensmusg2ensg = dict(zip(ortho_df["mouse_ensembl_id"].astype(str),
                        ortho_df["human_ensembl_id"].astype(str)))
print(f"  unique ENSMUSG -> ENSG: {len(ensmusg2ensg)}")

# Build per-symbol ENSG mapping
sym2ensg = {}
for sym, ensmusg in sym2ensmusg.items():
    if ensmusg in ensmusg2ensg:
        sym2ensg[sym] = ensmusg2ensg[ensmusg]
n_to_ensg = len(sym2ensg)
print(f"\n  BCG symbols mappable to ENSG: {n_to_ensg} / {len(bcg_symbols)} ({n_to_ensg/len(bcg_symbols):.1%})")

Loaded ortholog table: 14451 rows
  columns: ['human_ensembl_id', 'human_gene_name', 'mouse_ensembl_id', 'mouse_gene_name', 'orthology_type']
  one2one only: 14451 rows
  unique ENSMUSG -> ENSG: 14451

  BCG symbols mappable to ENSG: 9521 / 10866 (87.6%)


## 4. For each flavor: subset BCG to **intersection**(atlas HVG, BCG), normalize, write

Only genes that exist in BCG are kept; output width is **not** forced to 1,000.

In [ ]:
def make_bcg_atlas_hvg_intersection(bcg, sym2ensg, atlas_hvg_path, out_path):
    """Subset BCG to atlas HVG genes that exist in BCG — same column order as atlas list, skip missing."""
    atlas = sc.read_h5ad(atlas_hvg_path)
    atlas_hvg_genes = list(atlas.var_names.astype(str))
    n_target = len(atlas_hvg_genes)
    print(f"  atlas HVG: {n_target} ENSG IDs")
    ensg2sym = {ensg: sym for sym, ensg in sym2ensg.items()}
    bcg_var_idx = {s: i for i, s in enumerate(bcg.var_names.astype(str))}
    present_ensg = []
    bcg_col_idx = []
    for ensg in atlas_hvg_genes:
        sym = ensg2sym.get(ensg)
        if sym is not None and sym in bcg_var_idx:
            present_ensg.append(ensg)
            bcg_col_idx.append(bcg_var_idx[sym])
    n_present = len(present_ensg)
    print(f"  BCG intersection: {n_present} / {n_target} ({n_present/n_target:.1%}) — columns are real measurements only")
    if n_present == 0:
        raise RuntimeError("No atlas HVG genes found in BCG after mapping.")
    X_sub = bcg.X[:, bcg_col_idx]
    new_var = pd.DataFrame(index=pd.Index(present_ensg, name="ensg"))
    new_obs = bcg.obs.copy()
    new_a = ad.AnnData(X=X_sub, obs=new_obs, var=new_var)
    sc.pp.normalize_total(new_a, target_sum=1e4)
    sc.pp.log1p(new_a)
    keep_cols = [c for c in ["condition", "species", "cell_type", "study", "_scvi_batch"] if c in new_a.obs.columns]
    new_a = ad.AnnData(X=new_a.X, obs=new_a.obs[keep_cols].copy(), var=new_a.var.copy())
    new_a.write_h5ad(out_path)
    print(f"  wrote {out_path}: shape {new_a.shape}, .X mean={float(new_a.X.mean()):.4f}, max={float(new_a.X.max()):.4f}")
    return n_present, n_target


coverage_rows = []
for flavor in FLAVORS:
    print(f"\n=== flavor: {flavor} ===")
    atlas_path = os.path.join(DATASET_DIR, f"hvg_{flavor}_atlas_full_v07.h5ad")
    out_path = os.path.join(DATASET_DIR, f"bcg_mouse_atlashvg_intersect_{flavor}.h5ad")
    n_pres, n_tot = make_bcg_atlas_hvg_intersection(bcg, sym2ensg, atlas_path, out_path)
    coverage_rows.append({
        "flavor": flavor,
        "n_genes_intersection": n_pres,
        "n_atlas_hvg": n_tot,
        "coverage_pct": 100 * n_pres / n_tot,
        "out_path": out_path,
    })

coverage_df = pd.DataFrame(coverage_rows)
cov_csv = os.path.join(OUT_DIR, "bcg_atlas_hvg_coverage_intersection.csv")
coverage_df.to_csv(cov_csv, index=False)
print("\n=== Intersection summary (written", cov_csv, ") ===")
print(coverage_df.to_string(index=False))


## 5. Round-trip via CellOT env (`*_intersect_*_v07.h5ad`)


In [ ]:
files = [os.path.join(DATASET_DIR, f"bcg_mouse_atlashvg_intersect_{f}.h5ad") for f in FLAVORS]
v07_paths = []
for src in files:
    dst = src.replace(".h5ad", "_v07.h5ad")
    if os.path.exists(dst):
        os.remove(dst)
    shutil.copy2(src, dst)
    v07_paths.append(dst)

strip_script = r"""
import sys, h5py
EMPTY = ["layers","obsm","obsp","uns","varm","varp"]
for p in sys.argv[1:]:
    with h5py.File(p, "r+") as f:
        for g in EMPTY:
            if g in f and len(f[g].keys())==0:
                del f[g]
        for a in ("encoding-type","encoding-version"):
            if a in f.attrs:
                del f.attrs[a]
"""

rewrite_script = r"""
import sys, os, h5py, numpy as np, pandas as pd, anndata as ad
from scipy import sparse
def _d(x): return x.decode() if isinstance(x,(bytes,np.bytes_)) else x
def load_obs(f):
    g = f["obs"]; idx = _d(g.attrs["_index"]) if "_index" in g.attrs else "index"
    index = [_d(x) for x in g[idx][:]]; cols={}
    for n in g.keys():
        if n==idx: continue
        node=g[n]
        if isinstance(node, h5py.Group) and "categories" in node and "codes" in node:
            cats=[_d(c) for c in node["categories"][:]]
            cols[n]=pd.Categorical.from_codes(node["codes"][:], categories=cats)
        else:
            arr=node[:]
            if arr.dtype.kind in ("O","S"):
                arr=np.array([_d(x) for x in arr])
            cols[n]=arr
    return pd.DataFrame(cols, index=pd.Index(index, name=idx))
def load_var(f):
    g = f["var"]; idx = _d(g.attrs["_index"]) if "_index" in g.attrs else "index"
    index = [_d(x) for x in g[idx][:]]
    cols={}
    for n in g.keys():
        if n==idx: continue
        node=g[n]
        arr = node[:]
        if arr.dtype.kind in ("O","S"):
            arr=np.array([_d(x) for x in arr])
        cols[n]=arr
    return pd.DataFrame(cols, index=pd.Index(index, name=idx))
def load_X(f):
    n=f["X"]
    if isinstance(n,h5py.Group):
        d=n["data"][:]; i=n["indices"][:]; p=n["indptr"][:]
        sh=tuple(n.attrs.get("shape", n.attrs.get("h5sparse_shape")))
        e=_d(n.attrs.get("encoding-type", b"csr_matrix"))
        return sparse.csc_matrix((d,i,p),shape=sh) if "csc" in e else sparse.csr_matrix((d,i,p),shape=sh)
    return n[:]
for p in sys.argv[1:]:
    with h5py.File(p,"r") as f:
        obs=load_obs(f); var=load_var(f); X=load_X(f)
    a=ad.AnnData(X=X,obs=obs,var=var)
    os.remove(p); a.write(p)
    print("rewrote",p,"shape",a.shape)
"""

subprocess.run([CELLOT_PY, "-c", strip_script, *v07_paths], check=True)
subprocess.run([CELLOT_PY, "-c", rewrite_script, *v07_paths], check=True)
print("\nBCG intersection _v07 files (CellOT env):")
for p in v07_paths:
    print(f"  {p}")
